# Building micrograd

Environment check — run this first to confirm the kernel and libraries are wired up correctly.

In [20]:
import sys
import numpy as np
import matplotlib
import torch
import graphviz
import math 
import random 

# print(f"python    : {sys.version.split()[0]}")
# print(f"venv      : {sys.prefix}")
# print(f"numpy     : {np.__version__}")
# print(f"matplotlib: {matplotlib.__version__}")
# print(f"torch     : {torch.__version__}")
# print(f"graphviz  : {graphviz.__version__}")

In [21]:
class Value(): 

    def __init__(self, data, _children = (), _op = '', label = ''): 
        self.data = data 
        self.grad = 0.0 
        self._backward = lambda:None 
        self._prev = set(_children) 
        self._op = _op 
        self.label = label 
    
    def __repr__(self): 
        out = f"Value(data = {self.data})" 
        return out 
    
    def __add__(self, other): 
        other = other if isinstance(other, Value) else Value(other) 
        out = Value(self.data + other.data, (self, other), label = '+') 

        def _backward(): 
            self.grad += 1.0 * out.grad 
            other.grad += 1.0 * out.grad 
        out._backward = _backward 

        return out 
    
    def __radd__(self, other): 
        return self + other 

    def __mul__(self, other): 
        other = other if isinstance(other, Value) else Value(other) 
        out = Value(self.data * other.data, (self, other), label = '*') 

        def _backward(): 
            self.grad += other.data * out.grad 
            other.grad += self.data * out.grad 
        out._backward = _backward 

        return out 
    
    def __rmul__(self, other):
        return self * other 

    def __sub__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = self + (-1.0 * other)

        return out 

    def __rsub__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return self - other 
    
    def __pow__(self, other):
        assert(isinstance(other, (int, float)))
        out = Value(self.data ** other, (self, ), f'**{other}')

        def _backward():
            self.grad += out.grad * (other * (self.data ** (other - 1.0)))
        out._backward = _backward

        return out 
    
    def __rpow__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return pow(other, self)
    
    def __truediv__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * (other.data ** (-1.0)), (self, other), label = '/')
        
        def _backward():
            self.grad += (other.data ** (-1.0)) * out.grad 
            other.grad += (self.data * math.log(other.data)) * out.grad
        out._backward = _backward 

        return out 
    
    def __rtruediv__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return other / self

    def exp(self):
        out = Value(math.exp(self.data), (self, ), label = 'e')

        def _backward():
            self.grad += math.exp(self.data) * out.grad 
        out._backward = _backward

        return out 

    def tanh(self):
        x = self.data
        t = (math.exp(2*x) - 1)/(math.exp(2*x) + 1)
        out = Value(t, (self, ), 'tanh')

        def _backward():
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward

        return out
    
    def backward(self):
        topo = []
        visited = set()

        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        self.grad = 1.0

        for node in reversed(topo):
            node._backward()
    

In [22]:
from graphviz import Digraph

def trace(root):

  nodes, edges = set(), set()
  def build(v):
    if v not in nodes:
      nodes.add(v)
      for child in v._prev:
        edges.add((child, v))
        build(child)
  build(root)
  return nodes, edges

def draw_dot(root):
  dot = Digraph(format='svg', graph_attr = {'rankdir': 'LR'})
  nodes, edges = trace(root)
  for n in nodes:
    uid = str(id(n))

    dot.node(name = uid, label = "{%s | data %.4f | grad %.4f }" % (n.label, n.data, n.grad), shape = 'record')
    if n._op:

      dot.node(name = uid + n._op, label = n._op)

      dot.edge(uid + n._op, uid)
  for n1, n2 in edges:

    dot.edge(str(id(n1)), str(id(n2)) + n2._op)
  return dot


In [23]:
class Neuron():
    
    def __init__(self, nin):
        self.w = [Value(random.uniform(-1.0, 1.0)) for i in range(nin)]
        self.b = Value(random.uniform(-1.0, 1.0))

    def __call__(self, x):
        # act = w*x + b
        act = 0.0 

        for wi, xi in zip(self.w, x): 
            act += wi * xi 
        act += self.b 

        out = act.tanh() 
        return out 

    def parameters(self):
        return self.w + [self.b]

class Layer():

    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for i in range(nout)]
    
    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs 

    def parameters(self):
        out = []
        for neuron in self.neurons:
            for p in neuron.parameters:
                out.append(p)
        return out 

class MLP():

    def __init__(self, nin, nouts):
        sz = [nin] + nouts 
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]
    
    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x
    
    def parameters(self):
        out = []
        for layer in self.layers:
            for p in layer.parameters():
                out.append(p)
        return out 